### Execução do Fluxo de ELT

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName('etl_pipeline') \
    .config("spark.jars", "/opt/spark/jars/iceberg-spark-runtime-3.5_2.12-1.6.0.jar") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.spark_catalog.type", "hive") \
    .config("spark.sql.catalog.local.warehouse", "s3a://datalake/iceberg") \
    .getOrCreate()

#Ajuste de log WARN log para ERROR
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
## Instalar Bibliotecas
!pip install python-dotenv==1.0.1

In [ ]:
import os
from dotenv import load_dotenv, dotenv_values

import json
import requests

from pyspark.sql import DataFrame
from pyspark.sql.functions import explode, col, from_utc_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

In [ ]:
%run ../common/Common_env_functions.ipynb

In [ ]:
%run ../common/Common_data_functions.ipynb

### Carregar Variavel de Ambiente

In [ ]:
# list_env_var('../.env')
load_dotenv('../.env')
token = os.getenv("API_TOKEN")

### Endpoints da API 

In [ ]:
API_URL = 'https://api.olhovivo.sptrans.com.br/v2.1'
AUT = f"Login/Autenticar?token={token}"
LINHA = "Linha/Buscar?termosBusca="
L_POSICAO = "Posicao/Linha?codigoLinha="
L_PARADA = "Parada/BuscarParadasPorLinha?codigoLinha="
L_PREVISAO = "Previsao/Linha?codigoLinha="
EMPRESA = "Empresa"
POSICAO = 'Posicao'

id_linha = '8000'

### Schema da Tabela

In [ ]:
schema = [
    StructField("c", StringType(), True),
    StructField("cl", StringType(), True),
    StructField("sl", StringType(), True),
    StructField("lt0", StringType(), True),
    StructField("lt1", StringType(), True),
    StructField("qv", StringType(), True),
    StructField("vs", ArrayType(
        StructType([
            StructField("p", StringType(), True),
            StructField("a", StringType(), True),
            StructField("ta", StringType(), True),
            StructField("py", StringType(), True),
            StructField("px", StringType(), True),
            StructField("sv", StringType(), True),
            StructField("is", StringType(), True),
        ])
    ), True)
]

## Extrair os Dados

In [ ]:
# Orquestração Ingestão

try:
    auth_url = f"{API_URL}/{AUT}"

    session, authentication = auth(auth_url)

    if authentication:
        end_point = f"{API_URL}/{POSICAO}"

        data = fetch_data(session, authentication, end_point)
        
        if len(data):
            df_olhovivo = create_dataframe(schema, data['l'])
    
except Exception as e:
    print(f"Something went wrong: {e}") 

In [ ]:
print(f"Dados disponiveis:")
df_olhovivo.printSchema()

## Escrever os dados no Data Lake

In [ ]:
spark.sql("USE iceberg")

In [ ]:
(
    df_olhovivo
    .writeTo("iceberg.silver.tbl_silver_olhovivo")
    .createOrReplace()
)

In [ ]:
spark.sql("SHOW TABLES in silver").show()

## Transformar os Dados

In [ ]:
df_olhovivo_gold = spark.sql("""
    SELECT
     c AS Letreiro_Linha,
     cl AS Linha,
     sl AS Sentido,
     lt0 AS Destino_Linha,
     lt1 AS Origem_Linha,
     CAST(qv AS INT) AS Quantidade_Veiculos,
     vs
     
    FROM iceberg.silver.tbl_silver_olhovivo"""
).withColumn(
    "vs",
    explode(col("vs"))    
).select(
    "Letreiro_Linha",
    "Linha",
    "Sentido",
    "Destino_Linha",
    "Origem_Linha",
    "Quantidade_Veiculos",
    col("vs.p").alias("Prefixo_Veiculo"),
    col("vs.a").cast('boolean').alias("Veiculo_Acessivel"),
    from_utc_timestamp(col("vs.ta"),"America/Sao_Paulo").alias("Horario"),
    col("vs.py").alias("Latitude"),
    col("vs.px").alias("Longitude ")   
    
)

In [ ]:
df_olhovivo_gold.printSchema()

In [ ]:
(
    df_olhovivo_gold
    .writeTo("iceberg.gold.tbl_gold_olhovivo")
    .createOrReplace()
)

### Manutenção

In [ ]:
spark.sql("SHOW TABLES in gold").show()

In [ ]:
## Para deletar por completo do catalog e storage
# spark.sql("DROP TABLE iceberg.bronze.vendas PURGE")

In [ ]:
# spark.stop()